In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import torch
import torchaudio
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline


HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Missing HUGGINGFACE_TOKEN in .env")


if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required but not available.")

# -----------------------------
# 1. Whisper transcription
# -----------------------------
model = WhisperModel(
    "large-v3-turbo",
    device="cuda",
    compute_type="float16",
)

# -----------------------------
# 2. Speaker diarization
# -----------------------------
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN,
)
pipeline.to(torch.device("cuda"))

In [4]:

audio_path = r"/home/yteevm/Downloads/b2.mp3"
OUT_FILE = f"speaker_transcript_{os.path.basename(audio_path).replace('.mp3', '')}.txt"
segments, info = model.transcribe(
    audio_path,
    beam_size=5,
    vad_filter=True,
    language="en",
    word_timestamps=True,
)

segments = list(segments)
print(f"Detected language: {info.language} ({info.language_probability:.2%})")


waveform, sample_rate = torchaudio.load(audio_path)

diarization = pipeline({
    "waveform": waveform,
    "sample_rate": sample_rate,
})

annotation = diarization.speaker_diarization


Detected language: en (100.00%)


In [5]:

role_map = {
    "SPEAKER_00": "Host",
    "SPEAKER_01": "Marco Blume",
}

speaker_turns = []
for turn, _, speaker in annotation.itertracks(yield_label=True):
    speaker_turns.append({
        "start": float(turn.start),
        "end": float(turn.end),
        "speaker": speaker,
    })


# -----------------------------
# 3. Helper: find speaker at time
# -----------------------------
def get_speaker_for_time(t1: float, t2: float,turns: list[dict]) -> str:
    for turn in turns:
        if (turn["start"] <= t1 <= turn["end"]) and (turn["start"] <= t2 <= turn["end"]):
            return turn["speaker"]
    return "UNKNOWN"


# -----------------------------
# 4. Assign each word to speaker
# -----------------------------
word_items = []
word_items2 = []
prev_tmps = []
for seg in segments:
    if not getattr(seg, "words", None):
        continue

    for w in seg.words:
        if w.start is None or w.end is None:
            continue

        word_text = (w.word or "").strip()
        if not word_text:
            continue

        speaker = get_speaker_for_time(float(w.start), float(w.end), speaker_turns)


        tmp = {
            "start": float(w.start),
            "end": float(w.end),
            "speaker": speaker,
            "text": word_text,
        }


        if speaker == 'UNKNOWN':
            if not word_items:
                prev_tmps.append(tmp.copy())
            elif word_text[0] != word_text[0].lower():
                prev_tmps.append(tmp.copy())
            elif word_text[0] == word_text[0].lower():
                tmp['speaker'] = prev_speaker
                if prev_speaker != 'UNKNOWN':
                    word_items.append(tmp)

        else:
            for prev_tmp in prev_tmps:
                prev_tmp['speaker'] =  speaker
                word_items.append(prev_tmp)
            prev_tmps = []
            word_items.append(tmp)
        word_items2.append(tmp)
        prev_speaker= speaker


# -----------------------------
# 5. Merge consecutive words
#    from same speaker
# -----------------------------
merged = []

for item in word_items:
    if not merged:
        merged.append(item.copy())
        continue

    prev = merged[-1]

    # Merge if same speaker and close in time
    if (prev["speaker"] == item["speaker"]):
        prev["end"] = item["end"]
        prev["text"] += " " + item["text"]
    else:
        merged.append(item.copy())


# -----------------------------
# 6. Save transcript
# -----------------------------
with open(OUT_FILE, "w", encoding="utf-8") as f:
    for item in merged:
        f.write(
            f"[{item['start']:.2f} - {item['end']:.2f}] "
            f"{role_map[item['speaker']]}: {item['text']}\n"
        )

print(f"Saved to {OUT_FILE}")



Saved to speaker_transcript_b2.txt
